# Energy of world regions

!!! info ""
    :octicons-person-16: **[Pablo Rosado](https://ourworldindata.org/team/pablo-rosado)** • :octicons-calendar-16: August 21, 2026 *(last edit)* • [**:octicons-mail-16: Feedback**](mailto:info@ourworldindata.org?subject=Feedback%20on%20technical%20publication%20-%20Energy%20of%20world%20regions)

## Introduction

Our energy datasets combine three producers: the Energy Institute (EI) Statistical Review of World Energy, the U.S. Energy Information Administration (EIA), and Ember.
This document explains how our aggregates for [world regions](https://ourworldindata.org/grapher/continents-according-to-our-world-in-data) and income groups are built from them, and shows the evidence behind each choice.

It covers four issues:

* The Statistical Review reports much of the world inside residual regions like "Other Africa", whose country composition is undisclosed and varies by indicator.
* The two producers differ methodologically; most importantly, the EIA counts biofuels inside oil consumption.
* The Statistical Review's "other renewables" indicator does not exist in EIA data, and has to be derived from electricity generation.
* Stacked area charts can silently show part of an energy mix as though it were the whole.

## Initial setup
### Install and import requirements

To be able to run the code, there are some libraries you may need to install.

In [1]:
# Whether you run this code in your local computer or on Google Colab, you first need to install the required dependencies:
# * owid-catalog: a library that lets you explore and load datasets from OWID.
# * plotly: a library that lets you create interactive visualizations.
%pip install --upgrade owid-catalog plotly > /dev/null 2>&1

Note: you may need to restart the kernel to use updated packages.


Import the necessary libraries and define how data is loaded.

In [2]:
import json
from pathlib import Path

import pandas as pd
import plotly.express as px
from owid.catalog import Dataset, fetch


def load(path):
    # Load a table from OWID's public catalog.
    # When run inside a checkout of the ETL repository with the data built locally, the local files are
    # used instead (this is how the document was prepared before the data was published).
    for parent in [Path.cwd(), *Path.cwd().parents]:
        local = parent / "data" / Path(path).parent
        if local.exists():
            return Dataset(local)[Path(path).name].reset_index()
    return fetch(path).reset_index()

Set common variables.

In [3]:
# Fixed variables that ensure the analysis is run correctly and with the most recent data.
# There is no need to change them.
SR_VERSION = "2026-06-30"
EIA_VERSION = "2026-05-05"
ENERGY_MIX_VERSION = "2026-06-30"
REGIONS_VERSION = "2023-01-01"
CONTINENTS = ["Africa", "Asia", "Europe", "North America", "Oceania", "South America"]
INCOME_GROUPS = ["High-income countries", "Upper-middle-income countries", "Lower-middle-income countries", "Low-income countries"]
# Exajoules to terawatt-hours.
EJ_TO_TWH = 1e6 / 3600
# Colors used in all charts.
COLOR_EI, COLOR_ALT = "#0F6E67", "#A62F3B"

Load the data used throughout this document.

In [4]:
# Statistical Review (processed by OWID: country names harmonized, aggregates built).
tb_sr = load(f"garden/energy_institute/{SR_VERSION}/statistical_review_of_world_energy/statistical_review_of_world_energy")
# Statistical Review as published by EI (needed for its residual regions, which are excluded from the processed dataset).
tb_sr_raw = load(f"meadow/energy_institute/{SR_VERSION}/statistical_review_of_world_energy/statistical_review_of_world_energy")
# EIA international energy data (processed by OWID).
tb_eia = load(f"garden/eia/{EIA_VERSION}/international_energy/international_energy")
# The combined Energy mix dataset (Statistical Review extended with EIA at the country level).
tb_mix = load(f"garden/energy/{ENERGY_MIX_VERSION}/energy_mix/energy_mix")
# OWID region definitions.
tb_regions = load(f"garden/regions/{REGIONS_VERSION}/regions/regions").set_index("code")

## The Statistical Review's residual regions

The Statistical Review itemizes around 80 countries.
Everything else is reported inside residual regions such as "Other Africa" or "Other Caribbean": one number per indicator for all the remaining countries of an area.
Two properties make these regions problematic:

* **Their membership is undisclosed and varies by indicator.** "Other South America" covers whichever South American countries are not itemized for that indicator, so the same entity would quietly mean different things on different charts. For this reason we do not publish them as selectable entities.
* **Some straddle our region definitions.** "Other C.I.S." contains countries we assign to Europe (Moldova) and to Asia (Georgia, Armenia, Kyrgyzstan, Tajikistan), and "Other Asia Pacific" mixes Asian and Oceanian countries.

They must nonetheless be counted inside region aggregates, since otherwise those aggregates would miss every non-itemized country.
We assign each residual region to the region containing most of its energy: the African residuals to Africa, "Other Europe" to Europe, "Other C.I.S.", "Other Middle East" and "Other Asia Pacific" to Asia, "Other South America" to South America, and "Other Caribbean", "Central America" and "Other North America" to North America.

How much can these residual regions be trusted?
Each stands for a specific set of countries, and the EIA reports most of those countries individually, so the two can be compared directly.

In [5]:
# Countries that EI's "Definitions" sheet places in the Middle East and the C.I.S., and the two
# countries it places in Europe against our definitions.
MIDDLE_EAST = ["Iran", "Iraq", "Israel", "Jordan", "Kuwait", "Lebanon", "Oman", "Qatar", "Saudi Arabia", "Syria", "United Arab Emirates", "Yemen", "Bahrain", "Palestine"]
CIS = ["Russia", "Ukraine", "Belarus", "Moldova", "Armenia", "Azerbaijan", "Georgia", "Kazakhstan", "Kyrgyzstan", "Tajikistan", "Turkmenistan", "Uzbekistan"]
EI_EUROPE_EXTRA = ["Turkey", "Cyprus"]

# Historical entities are excluded from every comparison: the EIA reports East and West Germany or
# Czechoslovakia alongside their successors, which would count the same territory twice.
HISTORICAL = set(tb_regions.loc[tb_regions["is_historical"], "name"]) | {"Gibraltar"}


def region_members(name):
    codes = json.loads(tb_regions[tb_regions["name"] == name].iloc[0]["members"])
    return [c for c in tb_regions.loc[codes, "name"] if c not in HISTORICAL]


# The geography each residual region can stand for.
RESIDUAL_GEOGRAPHIES = {
    "Other South America": region_members("South America"),
    "Total Central America": region_members("Central America (UN M49)"),
    "Other Caribbean": region_members("Caribbean (UN M49)"),
    "Other Middle East": MIDDLE_EAST,
    "Other Africa": region_members("Africa"),
    "Other Asia Pacific": [c for c in region_members("Asia") if c not in MIDDLE_EAST + CIS + EI_EUROPE_EXTRA] + region_members("Oceania"),
}

# Total energy supply of each residual region, as published by EI, in TWh.
residuals = tb_sr_raw[tb_sr_raw["country"].isin(RESIDUAL_GEOGRAPHIES)][["country", "year", "tes_ej"]].copy()
residuals["Statistical Review's residual region"] = pd.to_numeric(residuals["tes_ej"], errors="coerce") * EJ_TO_TWH

# Total energy supply of the countries each residual region stands for, summed from EIA data.
# The total is the sum of the nine sources, on the same basis as the Statistical Review's.
EIA_SOURCES = ["energy_consumption_from_coal", "energy_consumption_from_petroleum", "energy_consumption_from_natural_gas", "energy_consumption_from_nuclear", "electricity_from_hydro", "electricity_from_solar", "electricity_from_wind", "energy_consumption_from_other_renewables", "energy_consumption_from_biofuels"]
eia_countries = tb_eia[~tb_eia["country"].str.contains("(EIA)", regex=False)].copy()
eia_countries["total"] = eia_countries[EIA_SOURCES].sum(axis=1, min_count=len(EIA_SOURCES))
itemized = set(tb_sr.loc[tb_sr["total_energy_supply_twh"].notna(), "country"])

compared = []
for bucket, geography in RESIDUAL_GEOGRAPHIES.items():
    stands_for = [c for c in geography if c not in itemized]
    proxy = eia_countries[eia_countries["country"].isin(stands_for)].groupby("year", observed=True)["total"].sum(min_count=1)
    a = residuals[residuals["country"] == bucket].set_index("year")["Statistical Review's residual region"]
    both = pd.DataFrame({"Statistical Review's residual region": a, "EIA, countries it stands for": proxy}).reset_index()
    both["region"] = f"{bucket} ({len(stands_for)} countries)"
    compared.append(both)
compared = pd.concat(compared, ignore_index=True).melt(id_vars=["year", "region"], var_name="series", value_name="value").dropna()

fig = px.line(compared, x="year", y="value", color="series", facet_col="region", facet_col_wrap=3, facet_row_spacing=0.12, color_discrete_sequence=[COLOR_EI, COLOR_ALT], title="Residual regions vs the EIA countries they stand for (total energy supply, TWh)")
fig.update_yaxes(matches=None, showticklabels=True, rangemode="tozero")
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_layout(legend_title_text="", legend=dict(orientation="h", y=-0.15), height=550)
fig.show()

On total energy supply, the residual regions agree with the EIA sum to within 3-14% in a typical year, which is the same order as the two producers' definitional differences for countries both report.
The Caribbean's early divergence reflects the EIA's incomplete coverage of small islands before 1990.

The Statistical Review also publishes a wider rollup, "Other South and Central America".
For most indicators it is exactly the sum of "Other South America", "Other Caribbean" and "Central America", which are already assigned, so it is redundant and ignored.
Where the finer regions are absent (reserves) or fall short of it (electricity generation by fuel, biodiesel), the difference is genuinely unassignable, and the affected aggregate is removed rather than published understated.

In [6]:
# Verify that the rollup decomposes exactly into the finer residual regions, on total energy supply.
rollup = pd.to_numeric(tb_sr_raw[tb_sr_raw["country"] == "Other S. & Cent. America"].set_index("year")["tes_ej"], errors="coerce")
parts = tb_sr_raw[tb_sr_raw["country"].isin(["Other South America", "Other Caribbean", "Total Central America"])]
parts = parts.groupby("year", observed=True)["tes_ej"].sum(min_count=1).reindex(rollup.index)
assert ((rollup - parts).abs() <= 1e-4 * rollup.abs()).all()

## Continents are taken from the producer

With every residual region assigned, the Statistical Review's continental aggregates cover the whole globe: the six continents sum to exactly its own World total, on every energy indicator.
We publish those aggregates directly, which preserves their full history (1965 onward).

The alternative would be to rebuild each continent by summing countries, extending the Statistical Review's ~80 with EIA data for the rest.
We measured that alternative before rejecting it: the two constructions differ by less than 2% in a typical year, and the rebuild could only start in 1980, when EIA coverage begins.

In [7]:
# Rebuild each continent by summing the country-level data of the combined Energy mix dataset.
mix_countries = tb_mix[~tb_mix["country"].isin(CONTINENTS + INCOME_GROUPS + ["World", "European Union (27)", "East Germany", "West Germany", "Czechoslovakia"])]
rebuilt = []
for region in CONTINENTS:
    members = region_members(region)
    s = mix_countries[mix_countries["country"].isin(members) & mix_countries["total_energy_supply_twh"].notna()]
    rebuilt.append(s.groupby("year", observed=True)["total_energy_supply_twh"].sum().rename(region))
rebuilt = pd.concat(rebuilt, axis=1).loc[1980:2024]

own = tb_sr[tb_sr["country"].isin(CONTINENTS)].pivot_table(index="year", columns="country", values="total_energy_supply_twh", observed=True)
both = pd.concat([own.melt(ignore_index=False, value_name="value").assign(series="Statistical Review's own aggregate"), rebuilt.melt(ignore_index=False, value_name="value").assign(series="Rebuilt from EI + EIA countries")]).reset_index().rename(columns={"country": "region", "variable": "region"})

fig = px.line(both, x="year", y="value", color="series", facet_col="region", facet_col_wrap=3, facet_row_spacing=0.12, color_discrete_sequence=[COLOR_EI, COLOR_ALT], title="Continents: the producer's aggregate vs a rebuild from countries (total energy supply, TWh)")
fig.update_yaxes(matches=None, showticklabels=True, rangemode="tozero")
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_layout(legend_title_text="", legend=dict(orientation="h", y=-0.15), height=550)
fig.show()

In [8]:
# Verify that the six continents add up to the producer's own World total, for total energy supply.
world = tb_sr[tb_sr["country"] == "World"].set_index("year")["total_energy_supply_twh"]
continents = tb_sr[tb_sr["country"].isin(CONTINENTS)].groupby("year", observed=True)["total_energy_supply_twh"].sum(min_count=len(CONTINENTS))
deviation = (100 * (continents - world) / world).dropna()
assert deviation.abs().max() < 0.1

## Oceania is the exception

Oceania is the one continent with no residual region of its own: "Other Asia Pacific" is folded into Asia, so Oceania's aggregate contains only the countries the Statistical Review names, and it names at most two (Australia and New Zealand).
Those two cover about 96% of the region's energy consumption; the remainder (mostly Papua New Guinea, New Caledonia and Fiji) is counted inside Asia.

For production the situation is worse: the Statistical Review reports no oil or gas production for New Zealand, so an Oceania production aggregate would be Australia alone, 7-18% below the region's level, with the gap growing after Papua New Guinea's gas fields came online.
We therefore require at least two reporting countries for an Oceania aggregate, which keeps every consumption indicator and withholds the production indicators.

In [9]:
oceania = []
for fuel, sr_col, eia_col in [("Oil", "oil_production_twh", "energy_production_from_petroleum"), ("Gas", "gas_production_twh", "energy_production_from_natural_gas")]:
    a = tb_sr[tb_sr["country"] == "Australia"].set_index("year")[sr_col].rename("Australia (Statistical Review)")
    b = tb_eia[tb_eia["country"] == "Oceania"].set_index("year")[eia_col].rename("Oceania (EIA, ~20 countries)")
    d = pd.concat([a, b], axis=1).reset_index().melt(id_vars="year", var_name="series", value_name="value").dropna()
    d["fuel"] = fuel
    oceania.append(d)
oceania = pd.concat(oceania, ignore_index=True)

fig = px.line(oceania, x="year", y="value", color="series", facet_col="fuel", color_discrete_sequence=[COLOR_EI, COLOR_ALT], title="Oceania's production: the only Oceanian producer EI reports vs the full region (TWh)")
fig.update_yaxes(matches=None, showticklabels=True, rangemode="tozero")
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_layout(legend_title_text="", legend=dict(orientation="h", y=-0.2), height=380)
fig.show()

## Income groups are rebuilt from countries

The Statistical Review assigns its residual regions to continents but to no income group, and no assignment is possible even in principle: the countries behind "Other South America" alone span three income groups.
Its income-group aggregates therefore cover only the itemized countries, reaching 91-99% of its own World total depending on the indicator; the low-income group is missing entirely.

We rebuild the four income groups by summing country-level data, combining the Statistical Review with the EIA for the countries it does not itemize.
Ember's electricity data, an independent third producer, confirms the defect: its lower-middle-income electricity generation exceeds the Statistical Review's by 14-16% in every year from 2000 to 2024.
Because most of the countries in these aggregates come from the EIA, whose coverage starts in 1980, the rebuilt income groups start in 1980.

In [10]:
panels = [("Lower-middle-income countries", "total_energy_supply_twh", "Lower-middle income: total energy supply"), ("Lower-middle-income countries", "hydro_consumption_twh", "Lower-middle income: hydropower"), ("Lower-middle-income countries", "other_renewables_consumption_twh", "Lower-middle income: other renewables"), ("High-income countries", "total_energy_supply_twh", "High income: total energy supply")]
MIX_COLUMN = {"total_energy_supply_twh": "total_energy_supply_twh", "hydro_consumption_twh": "hydro_twh", "other_renewables_consumption_twh": "other_renewables_twh"}
income = []
for region, col, title in panels:
    a = tb_sr[tb_sr["country"] == region].set_index("year")[col].rename("Statistical Review's own aggregate")
    b = tb_mix[tb_mix["country"] == region].set_index("year")[MIX_COLUMN[col]].rename("Rebuilt from EI + EIA countries")
    d = pd.concat([a, b], axis=1).reset_index().melt(id_vars="year", var_name="series", value_name="value").dropna()
    d["panel"] = title
    income.append(d)
income = pd.concat(income, ignore_index=True)

fig = px.line(income, x="year", y="value", color="series", facet_col="panel", facet_col_wrap=2, facet_row_spacing=0.14, color_discrete_sequence=[COLOR_EI, COLOR_ALT], title="Income groups: the producer's aggregates are systematically understated (TWh)")
fig.update_yaxes(matches=None, showticklabels=True, rangemode="tozero")
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_layout(legend_title_text="", legend=dict(orientation="h", y=-0.15), height=550)
fig.show()

## Combining the Statistical Review with EIA data

Extending the Statistical Review with EIA data raises the number of countries with a full nine-source energy mix from 80 to 230.
The Statistical Review is prioritized wherever both report a value, so no country's series mixes producers within a year.
Two methodological differences matter:

* **Different accounting bases for the total.** The EIA's own total energy nets out electricity trade and counts renewable electricity as generated rather than as heat input, so it is not comparable with its own by-source columns; our totals are always the sum of the nine sources.
* **The EIA counts biofuels as oil.** Its "petroleum and other liquids" consumption includes fuel ethanol and biodiesel blended into gasoline and diesel, while the Statistical Review's oil excludes them. We subtract the EIA's own biofuels figures from its oil consumption, so oil is a fossil fuel on both sides and biofuels are counted once, as a source of their own. Its oil *production* needs no correction: the EIA books biofuel output as renewable primary energy.

After the correction, the residual difference between the producers' oil figures for countries both report is around 5%, their ordinary definitional offset (natural gas liquids, refinery gain, bunker fuels).

In [11]:
oil = []
for country in ["Brazil", "United States"]:
    e = tb_eia[tb_eia["country"] == country].set_index("year")
    corrected = e["energy_consumption_from_petroleum"]
    # Adding back the biofuels we subtracted reconstructs the series as the EIA publishes it.
    published = corrected + e["energy_consumption_from_biofuels"].fillna(0)
    s = tb_sr[tb_sr["country"] == country].set_index("year")["oil_consumption_twh"]
    d = pd.concat([published.rename("EIA, petroleum and other liquids (as published)"), corrected.rename("EIA, minus its own biofuels"), s.rename("Statistical Review, oil")], axis=1)
    d = d.reset_index().melt(id_vars="year", var_name="series", value_name="value").dropna()
    d["country"] = country
    oil.append(d)
oil = pd.concat(oil, ignore_index=True)

fig = px.line(oil[oil["year"] >= 1980], x="year", y="value", color="series", facet_col="country", color_discrete_sequence=[COLOR_ALT, "#D98E97", COLOR_EI], title="Oil consumption: the EIA's series includes biofuels; the Statistical Review's does not (TWh)")
fig.update_yaxes(matches=None, showticklabels=True, rangemode="tozero")
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_layout(legend_title_text="", legend=dict(orientation="h", y=-0.2), height=400)
fig.show()

## Deriving "other renewables" from EIA data

The Statistical Review reports "other renewables" (geothermal, biomass power, tide and wave) as input-equivalent primary energy, which the EIA does not publish.
We derive it from the EIA's electricity generation using the efficiency factors the Statistical Review documents in its own methodology: geothermal at 10% thermal efficiency, biomass at 33%, and tide and wave at 100%, like hydropower.
Validated against every country-year both producers report, the derived figure matches the Statistical Review's with a median difference of 9%.

In [12]:
m = tb_eia[["country", "year", "energy_consumption_from_other_renewables"]].merge(tb_sr[["country", "year", "other_renewables_consumption_twh"]], on=["country", "year"]).dropna()
m = m[~m["country"].str.contains("(EI)", regex=False)]
m = m[(m["other_renewables_consumption_twh"] > 0.5) & (m["energy_consumption_from_other_renewables"] > 0.5)]
median_diff = (100 * (m["energy_consumption_from_other_renewables"] - m["other_renewables_consumption_twh"]).abs() / m["other_renewables_consumption_twh"]).median()
assert median_diff < 15

fig = px.scatter(m, x="other_renewables_consumption_twh", y="energy_consumption_from_other_renewables", hover_name="country", hover_data=["year"], log_x=True, log_y=True, opacity=0.25, color_discrete_sequence=[COLOR_ALT], title=f"Other renewables, derived from EIA generation vs reported by the Statistical Review (TWh)<br><sup>Every overlapping country-year ({len(m):,} points, median difference {median_diff:.0f}%); the dotted line marks equality.</sup>")
fig.add_shape(type="line", x0=0.5, y0=0.5, x1=1200, y1=1200, xref="x", yref="y", line=dict(color="#444444", width=1, dash="dot"))
fig.update_layout(height=520, xaxis_title="Statistical Review (TWh)", yaxis_title="Derived from EIA generation (TWh)")
fig.show()

## Stacked charts get dedicated indicators

A stacked area chart assembles several indicators, and if one is missing for an entity-year while others are present, the chart shows part of a mix as though it were the whole.
Each individual indicator is accurate on its own, so the fix is not to delete data.
Instead, our stacked charts read dedicated chart-specific indicators, built from the standalone ones under three rules:

1. A gap that the entity's reported total leaves no room for is a zero confirmed by the producer, and is filled (Sri Lanka reports no nuclear at all, but its other sources add up to its total).
2. Any other row missing a source is blanked entirely (Oceania's 1985-1999 electricity stack covered a fifth of its own total).
3. A row reporting every source whose sum still deviates from the reported total by more than 5% is blanked too (the United Kingdom's sources before 1985 reach 77% of its own total).

Both resulting invariants (whole mix or nothing, and the mix adds up to the total) are asserted at build time, so a future data release that breaks them fails our pipeline instead of publishing a spurious stack.
The standalone indicators are untouched, so single-source charts keep every accurate value.

## Known limitations

* Oceania's oil and gas production are not published: no source provides them for the full region without double counting, since New Zealand's and Papua New Guinea's production sits inside a residual region assigned to Asia.
* Oceania's consumption aggregates are understated by roughly 4-5%, the share of the region's energy inside "Other Asia Pacific".
* The rebuilt income groups start in 1980, and the low-income group depends almost entirely on EIA data.
* The membership the Statistical Review assumes for "Other Europe" and "Other C.I.S." is not published and cannot be verified against the EIA.
* Electricity generation by fuel has no source for income groups before 2000, so those aggregates start where Ember's data does.